## Imports

In [18]:
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from collections import Counter

## Load and View Data

In [19]:
# Load dataset
df = pd.read_excel(
    r"E:\Desktop\CS4AL3\Project Progress Report\2048_ML_Project\data\data.xlsx"
)

feature_names = [
    "TileA1", "TileA2", "TileA3", "TileA4",
    "TileB1", "TileB2", "TileB3", "TileB4",
    "TileC1", "TileC2", "TileC3", "TileC4",
    "TileD1", "TileD2", "TileD3", "TileD4"
]

X_full = df[feature_names].values
y_full = df["Move"].values

# List all games in order
game_numbers = sorted(df["Game_Number"].unique().tolist())

# Use the first 80% of games for training, 20% for testing.
split_index = int(len(game_numbers) * 0.8)

train_games = game_numbers[:split_index]
test_games = game_numbers[split_index:]

train_mask = df["Game_Number"].isin(train_games)
test_mask = df["Game_Number"].isin(test_games)

print("Training games:", train_games)
print("Test games:", test_games)

X_tr = X_full[train_mask]
X_te = X_full[test_mask]
y_tr = y_full[train_mask]
y_te = y_full[test_mask]

scaler = StandardScaler().fit(X_tr)
X_tr_s = scaler.transform(X_tr)
X_te_s  = scaler.transform(X_te)

knn = KNeighborsClassifier(n_neighbors=5).fit(X_tr_s, y_tr)
y_pred = knn.predict(X_te_s)

print("Test accuracy:", round((y_pred == y_te).mean(), 3))



Training games: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160]
Test games: [161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200]
Test accuracy: 0.52


### Simple Baseline Comparison

In [20]:
# can_move: check whether a move is valid on the current board for the baseline.
def can_move(board, move):
    grid = board.reshape(4, 4)

    if move == "RIGHT":
        grid = grid[:, ::-1]
    elif move == "UP":
        grid = grid.T
    elif move == "DOWN":
        grid = grid.T[:, ::-1]

    for row in grid:
        for i in range(1, 4):
            if row[i] != 0 and (row[i - 1] == 0 or row[i - 1] == row[i]):
                return True

    return False

# Count the moves in the training data
move_counts = Counter({"LEFT": 0, "RIGHT": 0, "UP": 0, "DOWN": 0})
move_counts.update(y_tr)

# Put the most common move first
move_order = [move for move, count in move_counts.most_common()]
print("Move order:", move_order)

# Choose the first valid move for each test board
simple_preds = []

for board in X_te:
    valid_moves = []

    for move in move_order:
        if can_move(board, move):
            valid_moves.append(move)

    if not valid_moves:
        raise ValueError("This test board has no valid moves.")

    simple_preds.append(valid_moves[0])

simple_preds = np.array(simple_preds)

print("Simple baseline accuracy:", round((simple_preds == y_te).mean(), 3))

Move order: ['LEFT', 'UP', 'RIGHT', 'DOWN']
Simple baseline accuracy: 0.477


## Record

#### 5 games, 50steps:
Test accuracy: 0.48, Simple baseline accuracy: 0.66

#### 10 games, 50steps:
Test accuracy: 0.49, Simple baseline accuracy: 0.41

#### 50 games, 50steps:
Test accuracy: 0.548, Simple baseline accuracy: 0.454

#### 135 games, 50steps:
Test accuracy: 0.524, Simple baseline accuracy: 0.51

#### 200 games, 50steps:
Test accuracy: 0.52, Simple baseline accuracy: 0.477